In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import pandas as pd
from sklearn.model_selection import train_test_split

In [6]:
data=pd.read_excel("deep1.xlsx")
data.shape

(103000, 10)

In [7]:
# data cleaning
data.isna().sum()

Season                                      0
Age                                      2022
Childish diseases                           0
Accident or serious trauma                  0
Surgical intervention                       0
High fevers in the last year                0
Frequency of alcohol consumption            0
Smoking habit                               0
Number of hours spent sitting per day    2038
Diagnosis                                   0
dtype: int64

In [12]:
data.duplicated().sum()

np.int64(3010)

In [13]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103000 entries, 0 to 102999
Data columns (total 10 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   Season                                 103000 non-null  object 
 1   Age                                    100978 non-null  float64
 2   Childish diseases                      103000 non-null  object 
 3   Accident or serious trauma             103000 non-null  object 
 4   Surgical intervention                  103000 non-null  object 
 5   High fevers in the last year           103000 non-null  object 
 6   Frequency of alcohol consumption       103000 non-null  object 
 7   Smoking habit                          103000 non-null  object 
 8   Number of hours spent sitting per day  100962 non-null  float64
 9   Diagnosis                              103000 non-null  object 
dtypes: float64(2), object(8)
memory usage: 7.9+ MB


In [18]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
data_no = data[["Age", "Number of hours spent sitting per day"]]

In [26]:
from sklearn.pipeline import Pipeline
mypipeline = Pipeline([ 
    ("impute", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])
scaled_array=mypipeline.fit_transform(data_no)
scaled_data = pd.DataFrame(scaled_array, columns=data_no.columns, index=data_no.index)
scaled_data.head()

,Age,Number of hours spent sitting per day
0,0.063987,-0.192358
1,0.342112,-0.145069
2,0.059735,-0.010464
3,-0.070889,-0.164136
4,-0.073511,-0.078501


In [29]:
data_cat = data.drop(columns=["Age", "Number of hours spent sitting per day"])

In [30]:
data_cat.head()

,Season,Childish diseases,Accident or serious trauma,Surgical intervention,High fevers in the last year,Frequency of alcohol consumption,Smoking habit,Diagnosis
0,spring,yes,yes,yes,more than 3 months ago,several times a week,daily,Normal
1,spring,yes,yes,yes,more than 3 months ago,several times a week,never,Normal
2,winter,yes,no,yes,no,several times a week,occasional,Normal
3,winter,yes,no,no,more than 3 months ago,hardly ever or never,never,Normal
4,winter,no,no,yes,more than 3 months ago,several times a week,occasional,Altered


In [34]:
from sklearn.preprocessing import OneHotEncoder
mypipeline1=Pipeline([
    (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        )
])
encoded_array=mypipeline1.fit_transform(data_cat)
encoded_cols = mypipeline1.named_steps["encoder"].get_feature_names_out(
    data_cat.columns
)
encoded_data = pd.DataFrame(encoded_array, columns=encoded_cols)

In [35]:
encoded_data.head()

,Season_fall,Season_spring,Season_summer,Season_winter,Childish diseases_no,Childish diseases_yes,Accident or serious trauma_no,Accident or serious trauma_yes,Surgical intervention_no,Surgical intervention_yes,...,Frequency of alcohol consumption_every day,Frequency of alcohol consumption_hardly ever or never,Frequency of alcohol consumption_once a week,Frequency of alcohol consumption_several times a day,Frequency of alcohol consumption_several times a week,Smoking habit_daily,Smoking habit_never,Smoking habit_occasional,Diagnosis_Altered,Diagnosis_Normal
0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [48]:
merge=pd.merge(scaled_data,encoded_data,left_index=True, right_index=True, how="outer")

np.int64(3010)

In [37]:
merge.head()

,Age,Number of hours spent sitting per day,Season_fall,Season_spring,Season_summer,Season_winter,Childish diseases_no,Childish diseases_yes,Accident or serious trauma_no,Accident or serious trauma_yes,...,Frequency of alcohol consumption_every day,Frequency of alcohol consumption_hardly ever or never,Frequency of alcohol consumption_once a week,Frequency of alcohol consumption_several times a day,Frequency of alcohol consumption_several times a week,Smoking habit_daily,Smoking habit_never,Smoking habit_occasional,Diagnosis_Altered,Diagnosis_Normal
0,0.063987,-0.192358,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,0.342112,-0.145069,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
2,0.059735,-0.010464,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,-0.070889,-0.164136,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,-0.073511,-0.078501,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [38]:
merge.shape

(103000, 25)

In [40]:
merge.duplicated().sum()

np.int64(3010)

In [42]:
merge1=merge.drop_duplicates()

In [43]:
merge1.shape

(99990, 25)

In [92]:
merge1 = merge1.drop(
    columns=["Diagnosis_Altered", "Diagnosis_Normal"], errors="ignore"
)
data=data.drop_duplicates()
ad = data["Diagnosis"]
ad = ad.to_frame()
ad_numeric = ad.copy()
ad_numeric["Diagnosis"] = ad_numeric["Diagnosis"].map(
    {"Normal": 0, "Altered": 1}
)


In [93]:
# model is ready to get trained 
X_train, X_test, Y_train, Y_test=train_test_split(
   merge1,ad_numeric,
    test_size=2,
    random_state=42
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
input_dim = X_train.shape[1]
model = Sequential(
    [
        Dense(16, activation="relu", input_shape=(input_dim,)),  # Fixed here
        Dense(8, activation="relu"),
        Dense(1, activation="sigmoid"),  
    ]
)
model.compile(
    optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
)
model.fit(X_train, Y_train, epochs=10,batch_size=64,validation_split=0.2,verbose=1)

C:\Users\admin\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.9386 - loss: 0.1598 - val_accuracy: 0.9896 - val_loss: 0.0323
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9892 - loss: 0.0224 - val_accuracy: 0.9897 - val_loss: 0.0175
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9898 - loss: 0.0171 - val_accuracy: 0.9892 - val_loss: 0.0169
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9892 - loss: 0.0162 - val_accuracy: 0.9891 - val_loss: 0.0160
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9895 - loss: 0.0158 - val_accuracy: 0.9903 - val_loss: 0.0157
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9894 - loss: 0.0155 - val_accuracy: 0.9901 - val_loss: 0.0148
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9896 - loss: 0.0153 - val_accuracy: 0.9894 - val_loss: 0.0168
Epoch 8/10
 755/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9894 - loss: 0.0149

In [96]:
test_loss,test_acc=model.evaluate(X_test,Y_test)
print(f"Test accuracy: {test_acc:.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 1.0000 - loss: 6.4901e-05
Test accuracy: 1.0000
